# Generative AI 012 — Document Loaders

The first of RAG's four components. A loader turns any source into `Document`
objects.

> **A constraint worth knowing first.** Every named loader — `TextLoader`,
> `PyPDFLoader`, `DirectoryLoader`, `WebBaseLoader`, `CSVLoader` — lives in
> **`langchain_community`**, a separate install. If it is not present, Part A
> will say so and the rest of this notebook still runs: `Document` and
> `BaseLoader` are both in `langchain-core`.

| Part | What we check |
|---|---|
| A | what is installed here |
| B | the `Document` object — 4 fields, 2 that matter |
| C | a custom loader: implement **one** method, get four free |
| D | `load()` grows **8.0×** with the corpus; `lazy_load()` stays **flat** |

Needs `langchain-core`.

In [ ]:
import warnings, importlib, csv, io, inspect, tracemalloc
warnings.filterwarnings("ignore")

from typing import Iterator
from langchain_core.documents import Document
from langchain_core.document_loaders import BaseLoader

## Part A — What is available

In [ ]:
for mod, why in (("langchain_community", "the five named loaders"),
                 ("pypdf", "what PyPDFLoader needs underneath")):
    try:
        m = importlib.import_module(mod)
        print(f"{mod:<22} {getattr(m, '__version__', 'ok'):<10} ({why})")
    except ImportError:
        print(f"{mod:<22} {'NOT INSTALLED':<10} ({why})")

print()
print("In langchain_core, and enough for everything below:")
print("   ", Document)
print("   ", BaseLoader)

## Part B — The `Document` object

A loader's whole job is to produce these.

In [ ]:
doc = Document(
    page_content="Cricket is a bat-and-ball game played between two teams.",
    metadata={"source": "cricket.txt", "page": 0},
)

print("page_content:", doc.page_content)
print("metadata    :", doc.metadata)
print("all fields  :", sorted(type(doc).model_fields))
print("repr        :", repr(doc))

assert sorted(type(doc).model_fields) == ["id", "metadata", "page_content", "type"]

`page_content` is what gets split, embedded and searched. `metadata` is what
lets an answer say **where it came from** — the difference between a citation
and a guess.

## Part C — Writing a loader

In [ ]:
class CsvRowLoader(BaseLoader):
    """One Document per row - the shape CSVLoader produces."""
    def __init__(self, text, source="data.csv"):
        self.text, self.source = text, source

    def lazy_load(self) -> Iterator[Document]:        # the ONLY method we write
        for i, row in enumerate(csv.DictReader(io.StringIO(self.text))):
            yield Document(
                page_content="\n".join(f"{k}: {v}" for k, v in row.items()),
                metadata={"source": self.source, "row": i},
            )

CSV = "name,age,city\nAyesha,27,Colombo\nRohit,31,Mumbai\nMei,24,Tokyo\n"
loader = CsvRowLoader(CSV)

docs = loader.load()             # we never wrote load()
print(len(docs), "documents\n")
print(docs[0].page_content)
print(docs[0].metadata)

assert len(docs) == 3

In [ ]:
# What did inheriting BaseLoader actually give us?
for name in ("lazy_load", "load", "aload", "alazy_load", "load_and_split"):
    written = "we wrote this" if name == "lazy_load" else "inherited, free"
    print(f"   {name:<16} {written}")

print()
print("BaseLoader.load, in its entirety:")
print("   ", inspect.getsource(BaseLoader.load).strip().splitlines()[-1].strip())

# This is lesson 010's argument arriving somewhere concrete.

## Part D — `load()` against `lazy_load()`

Same job — count the characters in a corpus — and watch peak memory as the
corpus grows.

In [ ]:
class BigLoader(BaseLoader):
    def __init__(self, n, size): self.n, self.size = n, size
    def lazy_load(self) -> Iterator[Document]:
        for i in range(self.n):
            yield Document(page_content="x" * self.size, metadata={"row": i})

SIZE = 5000
rows = []
print(f"{'documents':>10}{'corpus':>10}{'load()':>12}{'lazy_load()':>14}")
for n in (500, 1000, 2000, 4000):
    loader = BigLoader(n, SIZE)

    tracemalloc.start()
    docs = loader.load()
    a = sum(len(d.page_content) for d in docs)
    eager = tracemalloc.get_traced_memory()[1]
    tracemalloc.stop(); del docs

    tracemalloc.start()
    b = 0
    for d in loader.lazy_load():
        b += len(d.page_content)
    lazy = tracemalloc.get_traced_memory()[1]
    tracemalloc.stop()

    assert a == b                     # same work, same answer
    rows.append((n, eager, lazy))
    print(f"{n:>10}{n*SIZE/1e6:>8.0f} MB{eager/1e6:>9.2f} MB{lazy/1e3:>11.1f} KB")

In [ ]:
first, last = rows[0], rows[-1]
print(f"8x the corpus ({first[0]} -> {last[0]} documents):")
print(f"   load()      peak grew {last[1]/first[1]:>5.1f}x")
print(f"   lazy_load() peak grew {last[2]/first[2]:>5.1f}x")

assert last[1] / first[1] > 6        # tracks the corpus
assert last[2] / first[2] < 1.5      # flat
print()
print("A difference in KIND, not degree. load() builds the whole list, so its")
print("peak tracks the corpus. lazy_load() holds ONE Document at a time.")

> **What that measured.** Peak Python allocations under `tracemalloc`, with
> documents generated rather than read from disk — which isolates memory
> behaviour from file I/O. Not a benchmark of any real loader. The *shape* of
> the two columns is the claim, not the exact kilobytes.

**The rule:** a few small files, `load()` is simpler. Many files or large ones,
`lazy_load()`.

## What to take away

- RAG has four components; **document loaders** are the first.
- A `Document` carries **`page_content`** and **`metadata`** — metadata is what
  makes a citation possible.
- Loaders differ in **what one Document means**: per file, per page, per row,
  per URL. That decides what a citation can point at.
- Write a loader by inheriting `BaseLoader` and implementing **`lazy_load`
  alone**; `load` is `list(self.lazy_load())`.
- **8× the corpus grew `load()`'s peak 8.0× and left `lazy_load()`'s flat.**

## Exercises

1. Add `encoding` handling to `CsvRowLoader` and a `TextLoader`-alike that
   returns one Document per file. Which is fewer lines?
2. Your loader yields `metadata={"row": i}`. What else would you want there
   before putting this into a RAG system you had to debug?
3. Part D generated documents. Write 4,000 small files to a temp directory and
   rerun it. Does file I/O change the shape of the two columns?
4. `load_and_split` was inherited free. Read its source — what does it assume,
   and when would that assumption be wrong?
5. If `langchain-community` is installed, load a real PDF with `PyPDFLoader`
   and confirm you get one Document per page. Then inspect `docs[0].metadata`
   and compare it with what your custom loader produces.